# Imports

In [13]:
import pickle
import optuna

import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [11]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004922,0.001773,0.993304,0.004134,0.002593,0.993273,0.004337,0.000774,0.994889
1,0.997228,0.000463,0.002309,0.995591,0.000405,0.004004,0.994889,0.001056,0.004055
2,0.002369,0.000604,0.997026,0.001958,0.000503,0.997538,0.003438,0.001069,0.995493
3,0.004656,0.003362,0.991982,0.002466,0.002391,0.995142,0.004847,0.002650,0.992503
4,0.995353,0.000016,0.004632,0.994178,0.000003,0.005819,0.996479,0.000171,0.003351


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.012452,0.003199,0.984349,0.011048,0.004491,0.984461,0.007391,0.003335,0.989274
1,0.472116,0.002331,0.525553,0.499186,0.001896,0.498918,0.531268,0.002217,0.466515
2,0.998690,0.001012,0.000298,0.997826,0.001594,0.000579,0.995687,0.003676,0.000637
3,0.980854,0.000018,0.019128,0.987434,0.000009,0.012557,0.995089,0.000291,0.004620
4,0.006765,0.001847,0.991388,0.006876,0.001972,0.991152,0.004747,0.003551,0.991701


# Machine Learning

In [14]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [42]:
def objective(trial, X, y):
    
    C = trial.suggest_float("C", 1e-5, 100.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
    tol = trial.suggest_float("tol", 1e-4, 1e-2, log=True)
    max_iter = trial.suggest_int("max_iter", 1000, 5000)

    w0 = trial.suggest_float('weight_class_0', 0.05, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.05, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.05, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        
        X_train_fold, X_valid_fold = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        model = LogisticRegression(
            solver="lbfgs",
            l1_ratio=0.0,
            C=C,
            class_weight=class_weight,
            fit_intercept=fit_intercept,
            tol=tol,
            max_iter=max_iter,
            random_state=42,
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)
        
        weighted_probas = proba * weights
        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=500, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-08 16:16:58,578] A new study created in memory with name: no-name-ded1b4db-1515-400a-9d5b-6c450f832994
Best trial: 0. Best value: 0.333333:   0%|▎                                                                                                                                      | 1/500 [00:11<1:39:06, 11.92s/it]

[I 2026-07-08 16:17:10,482] Trial 0 finished with value: 0.3333333333333333 and parameters: {'C': 1.933111004532516e-05, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.0038215785301761456, 'max_iter': 1619, 'weight_class_0': 0.25840460120773046, 'weight_class_1': 0.13117619260016686, 'weight_class_2': 1.5767405064909965}. Best is trial 0 with value: 0.3333333333333333.


Best trial: 8. Best value: 0.937338:   0%|▌                                                                                                                                      | 2/500 [00:17<1:07:22,  8.12s/it]

[I 2026-07-08 16:17:15,891] Trial 8 finished with value: 0.9373375156264157 and parameters: {'C': 0.00027347651254961894, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.004470907046487394, 'max_iter': 3774, 'weight_class_0': 0.11645254607351578, 'weight_class_1': 0.43189630013191777, 'weight_class_2': 0.11426748531792445}. Best is trial 8 with value: 0.9373375156264157.


Best trial: 8. Best value: 0.937338:   1%|█                                                                                                                                        | 4/500 [00:18<25:54,  3.13s/it]

...

[I 2026-07-08 16:33:47,815] Trial 495 finished with value: 0.9495954952542407 and parameters: {'C': 0.04852262895753875, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.00024292223310440933, 'max_iter': 2576, 'weight_class_0': 0.2824370874480394, 'weight_class_1': 1.7878342954700925, 'weight_class_2': 1.4438778838087711}. Best is trial 320 with value: 0.9497521962450051.


Best trial: 320. Best value: 0.949752: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 498/500 [16:49<00:02,  1.00s/it]

[I 2026-07-08 16:33:48,457] Trial 497 finished with value: 0.94925565438709 and parameters: {'C': 0.25319371502331134, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.00030262112912856335, 'max_iter': 4705, 'weight_class_0': 0.19432444497875603, 'weight_class_1': 1.1337130736343826, 'weight_class_2': 1.741145708636521}. Best is trial 320 with value: 0.9497521962450051.


Best trial: 320. Best value: 0.949752: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [16:50<00:00,  2.02s/it]

[I 2026-07-08 16:33:48,707] Trial 499 finished with value: 0.9493121221330758 and parameters: {'C': 0.053437951140203825, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.0003022478366219594, 'max_iter': 4648, 'weight_class_0': 0.18764628089020385, 'weight_class_1': 1.8668121836642209, 'weight_class_2': 1.6699953451770189}. Best is trial 320 with value: 0.9497521962450051.
[I 2026-07-08 16:33:48,857] Trial 498 finished with value: 0.9490095834950196 and parameters: {'C': 0.05934892750671261, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.0003002000598966914, 'max_iter': 4629, 'weight_class_0': 0.16015575303819574, 'weight_class_1': 1.8338006593817024, 'weight_class_2': 1.5502079022726156}. Best is trial 320 with value: 0.9497521962450051.

Best trial score:
0.9497521962450051

Best params:
{'C': 0.016583182054671262, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.0010484096989234488, 'max_iter': 2941, 'weight_class_0': 0.7574214432061076, 'weight_

In [43]:
lg_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

lg = LogisticRegression(
    solver="lbfgs",
    l1_ratio=0.0,
    random_state=42,
    **lg_params
).fit(X_train, y_train.health_condition)

test_proba = lg.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [44]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [45]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.4.1_xgboost_lightgbm_catboost_stacking_submission.csv', index=False)

In [46]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [47]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'xgb_0', 'xgb_1', 'xgb_2', 'cat_0',
       'cat_1', 'cat_2'],
      dtype='str')

In [48]:
len(study.trials)

500